In [25]:
import numpy as np
import pandas as pd
import datetime as dt

In [26]:
df = pd.read_csv("all_fights.csv")

In [27]:
df["fight_date"] = pd.to_datetime(df["fight_date"])

df = df.sort_values("fight_date").reset_index(drop=True)

In [28]:
print(df['red_winner'].dtype)
print(df['red_winner'].unique())

str
<StringArray>
['f', 't']
Length: 2, dtype: str


In [29]:
df["red_winner"] = df["red_winner"].map({
    "t": 1,
    "f": 0
})

In [30]:
elo = {}

K = 32

df["red_elo"] = np.nan
df["blue_elo"] = np.nan
df["elo_diff"] = np.nan

def expected_score(player, opponent):
    return 1 / (1 + 10 ** ((opponent - player) / 400))

for index, row in df.iterrows():

    red = row["red_fighter"]
    blue = row["blue_fighter"]

    red_rating = elo.get(red, 1500)
    blue_rating = elo.get(blue, 1500)

    df.loc[index, "red_elo"] = red_rating
    df.loc[index, "blue_elo"] = blue_rating
    df.loc[index, "elo_diff"] = red_rating - blue_rating

    expected_red = expected_score(red_rating, blue_rating)
    expected_blue = expected_score(blue_rating, red_rating)

    actual_red = row["red_winner"]
    actual_blue = 1 - actual_red

    new_red = red_rating + K * (actual_red - expected_red)
    new_blue = blue_rating + K * (actual_blue - expected_blue)

    elo[red] = new_red
    elo[blue] = new_blue

In [31]:
df[['red_elo', 'blue_elo', 'red_winner']].notna().sum()

red_elo       7265
blue_elo      7265
red_winner    7265
dtype: int64

In [32]:
df[['red_elo', 'blue_elo', 'red_winner']].tail(20)

,red_elo,blue_elo,red_winner
7245,1652.809498,1633.094091,0
7246,1616.148311,1571.610061,1
7247,1531.380181,1500.729390,1
7248,1498.685035,1487.326754,0
7249,1620.076045,1516.500593,1
7250,1542.425047,1463.361284,1
7251,1557.819330,1501.438774,1
7252,1549.308586,1536.182700,1
7253,1596.425356,1521.418364,0
7254,1533.013116,1549.122934,1


In [33]:
df_elo = df[['red_elo', 'blue_elo', 'red_winner']].copy()

In [34]:
df2 = pd.read_csv("processed_ufc_fights.csv")

In [37]:
df2[['red_elo', 'blue_elo', 'elo_diff']] = df[['red_elo', 'blue_elo', 'elo_diff']]

In [38]:
df2.tail(20)

,red_odds,blue_odds,red_winner,odds_diff,age_diff,reach_diff,height_diff,wins_diff,losses_diff,rounds_diff,...,red_reach_per_height,blue_reach_per_height,red_days_since_last,blue_days_since_last,red_debut,blue_debut,activity_diff,red_elo,blue_elo,elo_diff
7245,-590,390,0,-980,-3,-2.54,2.54,-8,-7,-61,...,1.013514,1.041096,266.0,77.0,0,0,189.0,1652.809498,1633.094091,19.715407
7246,-174,136,1,-310,3,5.08,7.62,2,3,21,...,1.012658,1.026316,196.0,105.0,0,0,91.0,1616.148311,1571.610061,44.538249
7247,-670,430,1,-1100,-3,10.16,5.08,1,-1,-3,...,1.067568,1.041667,175.0,203.0,0,0,-28.0,1531.380181,1500.729390,30.650791
7248,158,-205,0,363,4,-2.54,-2.54,2,0,2,...,1.000000,1.000000,210.0,287.0,0,0,-77.0,1498.685035,1487.326754,11.358280
7249,-156,122,1,-278,-1,15.24,7.62,7,-1,18,...,1.028571,0.985075,154.0,196.0,0,0,-42.0,1620.076045,1516.500593,103.575452
7250,-330,240,1,-570,0,0.00,2.54,4,-5,-7,...,1.014286,1.028986,70.0,371.0,0,0,-301.0,1542.425047,1463.361284,79.063763
7251,-1100,600,1,-1700,-11,5.08,0.00,4,-2,2,...,1.065789,1.039474,105.0,427.0,0,0,-322.0,1557.819330,1501.438774,56.380556
7252,240,-330,1,570,5,7.62,-2.54,18,11,63,...,1.044118,0.985507,392.0,238.0,0,0,154.0,1549.308586,1536.182700,13.125886
7253,144,-186,0,330,1,5.08,7.62,7,2,18,...,1.026667,1.041667,210.0,147.0,0,0,63.0,1596.425356,1521.418364,75.006992
7254,136,-174,1,310,4,7.62,7.62,1,2,5,...,1.000000,1.000000,525.0,224.0,0,0,301.0,1533.013116,1549.122934,-16.109817


In [39]:
df2.to_csv(
    "processed_ufc_fights.csv",
    index=False
)